## SAE Feature Analysis

Loads precomputed results from `scripts/analyze_features.py` and produces:
1. Lift matrix heatmap (features x labels)
2. Co-activation matrix with hierarchical clustering
3. Per-label boolean composition tree visualization
4. Minimal-feature-set AUROC curves per label
5. Temporal enrichment scatter (early vs late activation fraction)

In [ ]:
import json
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path

matplotlib.use("module://matplotlib_inline.backend_inline")
plt.rcParams.update({
    "figure.dpi": 150,
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
})

-- Config --

In [ ]:
MODEL        = "test_01"
EXPERIMENTS  = Path("experiments")
MODEL_DIR    = EXPERIMENTS / MODEL
ANALYSIS_DIR = MODEL_DIR / "analysis"
FIGURES_DIR  = ANALYSIS_DIR / "features" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

-- Load results --

In [ ]:
with open(ANALYSIS_DIR / "features.json") as f:
    results = json.load(f)

npz = dict(np.load(ANALYSIS_DIR / "features.npz", allow_pickle=True))

lift_matrix       = npz["lift_matrix"]           # (n_feat, n_labels)
coactivation_mat  = npz["coactivation_matrix"]   # (n_active, n_active)
coact_indices     = npz["coactivation_indices"].tolist()
spec_indices      = npz["specificity_indices"].tolist()
label_names       = npz["label_names"].tolist()

print(f"Lift matrix: {lift_matrix.shape}")
print(f"Co-activation matrix: {coactivation_mat.shape}")
print(f"Labels: {label_names}")

### 1. Feature-Label Lift Matrix

Rows = SAE features (top by max lift across labels), columns = labels.
Values show P(label=1 | feature active) / P(label=1).
Lift > 1 means the feature enriches for that label.

In [ ]:
from src.analysis.plotting import plot_lift_heatmap, show_or_savefig

plot_lift_heatmap(
    lift_matrix, label_names, spec_indices,
    max_features=40,
    show=True,
    save_path=FIGURES_DIR / "lift_heatmap",
)

### 2. Co-Activation Matrix (Hierarchically Clustered)

Entry $(i,j) = \frac{P(i\text{ AND }j\text{ active})}{(P(i) * P(j))}$.
Values > 1 indicate features that co-fire more often than chance.
Dendrogram groups features with similar co-activation patterns.

In [ ]:
from src.analysis.plotting import plot_coactivation_matrix

plot_coactivation_matrix(
    coactivation_mat, coact_indices,
    max_features=50,
    show=True,
    save_path=FIGURES_DIR / "coactivation_clustered",
)

### 3. Boolean Composition Rules

For each label, a shallow decision tree extracts boolean AND/OR rules
over SAE features. The compositional gap = tree AUROC - best single
feature AUROC measures how much the concept requires feature combinations.

In [ ]:
from src.analysis.plotting import plot_composition_rules

composition = results.get("composition", {})
if composition:
    plot_composition_rules(
        composition,
        show=True,
        save_path=FIGURES_DIR / "composition_rules",
    )
else:
    print("No composition results found")

### 4. Minimal Feature Set AUROC Curves

Greedy forward selection: at each step, add the feature that maximally
improves AUROC. The curve shows how quickly each label's concept can
be recovered from SAE features. Fewer features = more monosemantic.

In [ ]:
from src.analysis.plotting import plot_minimal_feature_curves

minimal = results.get("minimal_feature_set", {})
if minimal:
    plot_minimal_feature_curves(
        minimal,
        show=True,
        save_path=FIGURES_DIR / "minimal_feature_auroc",
    )
else:
    print("No minimal feature set results found")

### 5. Temporal Enrichment

Each point is an SAE feature. X-axis = activation rate in the earliest
quartile of encounter times; Y-axis = activation rate in the latest
quartile. Features far from the diagonal activate preferentially at
the beginning or end of patient trajectories. Color = |Pearson r|
between activation magnitude and absolute time.

In [ ]:
from src.analysis.plotting import plot_temporal_enrichment

temporal = results.get("temporal", [])
if temporal:
    plot_temporal_enrichment(
        temporal,
        show=True,
        save_path=FIGURES_DIR / "temporal_enrichment",
    )
else:
    print("No temporal enrichment results found")